# 📜 Translations

This notebook explains how translations are sourced, how to download them, and how to use them in the parsing pipeline.

**Important:** Translations come from the ORACC website and represent scholarly interpretations of the texts. They are stored separately from the cuneiform data and are completely unaffected by parsing configuration options (`max_break_fraction`, `drop_missing`, `drop_damaged`, `mask_pos`).

## 🎯 Goals

1. **Understand the source** — where translations come from and what they contain
2. **Getting the translation data** — fetch pre-cached translations from Zenodo
3. **Parse with translations** — enable and access translations in the pipeline through ORACC servers
4. **Understand the relationship to parsing config** — why translations can show text that the cuneiform output does not

## Prerequisite: install the package

In [1]:
# Install oracc-parser from PyPI
!pip install oracc-parser -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.4/140.4 kB 3.1 MB/s eta 0:00:00


## Prerequisite: set up the data directory

A DATA_DIR must be defined for the package to work correctly. This is where the data is stored after download from Zenodo or the live ORACC server.

**Option 1**: Colab

Set it up as a folder in your Google Drive (requires mounting Google Drive).

**Option 2**: Run Locally

Set it up anywhere you want on your computer.

Choose whichever option below you prefer (comment out the other one).

In [2]:
# --- Option 1: persist downloaded data across Colab sessions using Google Drive ---
# Without this, data downloads to /content/oracc_data and is lost when the session ends.
# Uncomment the lines below to mount your Drive and store data there persistently.
#
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ["ORACC_DATA_DIR"] = "/content/drive/MyDrive/oracc_data"

# --- Option 2: run locally (not in Colab) ---
# If you are running this notebook locally, set the data directory here.
# Uncomment the lines below and set the path to where you want data stored.
#
# import oracc_parser.settings as settings
# from pathlib import Path
# settings.DATA_DIR = Path("/path/to/your/oracc_data")

Mounted at /content/drive


## 1. Source of Translations

For many ORACC projects, scholars have produced primarily English (more rarely German) translations that are published alongside the cuneiform text on the ORACC website. There are two ways to access these translations using the package:

1. Download translations for all the projects at once through the Zenodo repository (better for big corpus)
2. Fetching translations from the ORACC servers through HTTP requests for each tablet individually (better for small corpus)

A few things to keep in mind:

- **Not all tablets have translations.** Coverage varies by project — some projects are fully translated, others have none. Tablets without a translation return an empty string.
- **Translations are scholarly reconstructions.** They reflect the editor's interpretation of the full text, including damaged or missing sections. A tablet that is 30% broken may still have a complete translation.
- **Translations are not stored in the word CSVs.** There is no alignment between the cuneiform texts at the word level and their translations.

## 2. Getting the translation data

There are two ways to get translations.

### Option 1 — Download the pre-cached archive from Zenodo (recommended)

The Zenodo record includes `oracc_html_translations.zip`, a 130 MB archive of pre-downloaded HTML pages for all 25,994 tablets in the dataset. Download it once and all subsequent translation lookups are instant (no network requests).

In [8]:
from oracc_parser.download.fetch_data import fetch_data
from oracc_parser.settings import TRANSLATIONS_DIR

fetch_data(include_translations=True)
n_cached = len(list(TRANSLATIONS_DIR.rglob("*.html")))
print(f"Done: {n_cached} HTML pages now cached")

Fetching file list from https://zenodo.org/records/20625379...
  oracc_html_translations.zip (130.0 MB)
  catalogues.zip (2.7 MB)
  ✓ oracc_html_translations.zip already extracted, skipping
  ✓ catalogues.zip already extracted, skipping

✓ Done. Data is ready in /content/drive/MyDrive/oracc_data
Done: 25903 HTML pages now cached


### Option 2 — Fetch live from ORACC (for projects not in Zenodo)

Translations can be fetched automatically from the ORACC website the first time each tablet is parsed. The HTML page is then saved to the local cache, so subsequent runs are instant.

This happens when `fetch_translations=True` in `RunConfig` (the default is `False`). No extra steps are needed, but be aware that parsing a large project for the first time will make one HTTP request per tablet.

## 3. Parsing with translations

Set `fetch_translations=True` in `RunConfig`.

If the Zenodo cache is present, this reads from disk and adds no noticeable overhead.

If the Zenodo cache is not present, it makes HTTP requests per text to get the translations.

In [4]:
from oracc_parser import parse_project, RunConfig

PROJECT = "borsippa"
config = RunConfig(fetch_translations=True, limit=5)
records = parse_project(PROJECT, config=config)

n_with_translation = sum(1 for r in records if r.content.english_translation)
print(f"Parsed {len(records)} tablets — {n_with_translation} have a translation")

17:52:21 | INFO    | oracc_parser | Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/borsippa
INFO:oracc_parser:Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/borsippa
Processing borsippa: 100%|██████████| 5/5 [00:01<00:00,  3.47it/s]
17:52:23 | INFO    | oracc_parser | Processed 5 tablets for borsippa from word CSVs
INFO:oracc_parser:Processed 5 tablets for borsippa from word CSVs


Parsed 5 tablets — 5 have a translation


In [5]:
# Find a tablet that has a translation and print it
for record in records:
    if record.content.english_translation:
        print(f"=== {record.metadata.id_text} ===")
        print()
        print("TRANSLITERATION (first 300 chars):")
        trans = record.content.transliterated_str_representation
        print(trans.text[:300] if trans else "(none)")
        print()
        print("ENGLISH TRANSLATION:")
        print(record.content.english_translation)
        break

=== P202111 ===

TRANSLITERATION (first 300 chars):
2 1/2 u₄-mu maš-ti-ti ša₂ {iti}GUD 2 1/2 u₄-mu
[maš]-ti-ti {iti}KIN 2 1/2 u₄-mu kap-ri ša₂ {iti}GAN
1 u₄]-mu maš-ti-ti ša₂ {iti}ZIZ₂ 2 u₄-mu maš-ti-ti ša₂ {iti}ŠE
[PAB] 10 1/2 u₄-mu kap-ri u₃ maš-ti-ti
[i]-⸢na⸣ GIŠ.ŠUB.BA ša₂ {m}{d}EN-SUM{na} DUMU-šu₂ ša₂ {m}{d}AG-EDURU-MU DUMU {m}DINGIR-ia₂
[a-di] 

ENGLISH TRANSLATION:
2½ days maštītu in month II
2½ days maštītu (in) month VI
2½ days kapru in month IX
[1] day maštītu in month XI
2 days maštītu in month XII
[in total] 10½ days kapru and maštītu [of (?)] the prebend of Bēl-iddin/Nabû-aplu-iddin/Ilia: (these days) are at the disposal of Marduk-šumu-ibni for performing (ana ēpišānūti) until the end of the agreement.
Marduk-šumu-ibni [shall measure out (barley and dates)] from the bīt-asê.
[He is responsible for] approaching the meal, [for uninterrupted service and punctuality]. [ (...) ]
Marduk-šumu-ibni has not received [ x ] [fro]m Bēl-iddin.
Witnesses: Nabû-uballiṭ/Nabû-šumu-iddin/Il

Once `fetch_translations=True`, when running `get_full_flat_table` you will also get a translation column.

In [7]:
from oracc_parser import get_full_flat_table
# Full flat table — transliteration, normalization, lemmatization, unicode, and translation in one DataFrame
flat = get_full_flat_table(records)
print(f"Columns: {list(flat.columns)}")
flat.head()

Columns: ['id', 'project', 'text_id', 'genre', 'archive', 'provenance', 'pleiades_id', 'state_supergroup', 'period', 'start_year', 'end_year', 'accession_museum_publication_numbers', 'secondary_literature', 'credits', 'cite_as', 'transliteration', 'normalization', 'lemmatization', 'unicode', 'translation', 'total_tokens', 'tokens_without_broken']


,id,project,text_id,genre,archive,provenance,pleiades_id,state_supergroup,period,start_year,...,secondary_literature,credits,cite_as,transliteration,normalization,lemmatization,unicode,translation,total_tokens,tokens_without_broken
0,borsippa_P202111,borsippa,P202111,Legal,Ilia Archive,Borsippa,893964,Mix,achaemenid,-547,...,,"Adapted from Caroline Waerzeggers, The Ezida T...",Please cite this page as http://oracc.org/bors...,2 1/2 u₄-mu maš-ti-ti ša₂ {iti}GUD 2 1/2 u₄-mu...,UNK UNK ūmu maštīti ša Ayyaru UNK UNK ūmu\nmaš...,UNK UNK ūmu maštītu ša Ayyaru UNK UNK ūmu\nmaš...,𒈫 𒈦 𒌓𒈬 𒈦𒋾𒋾 𒃻 𒌗𒄞 𒈫 𒈦 𒌓𒈬\n𒈦𒋾𒋾 𒌗𒆥 𒈫 𒈦 𒌓𒈬 𒆏𒊑 𒃻 𒌗𒃶\...,2½ days maštītu in month II\n2½ days maštītu (...,102,101
1,borsippa_P202351,borsippa,P202351,Legal,Ilia Archive,Borsippa,893964,Mix,Neo-Babylonian,-626,...,,"Adapted from Caroline Waerzeggers, The Ezida T...",Please cite this page as http://oracc.org/bors...,{[m}šu-la-a A-šu₂ ša₂] {m}ṣil-la-a A {m}DINGIR...,Šula māršu ša Ṣilla mār Ilia\nina hūd libbišu ...,Šula māru ša Ṣilla māru Ilia\nina hūdu libbu K...,𒁹𒋗𒆷𒀀 𒀀𒋙 𒃻 𒁹𒉣𒆷𒀀 𒀀 𒁹𒀭𒅀\n𒀸 𒄷𒌓 𒊮𒁉𒋙 𒁹𒆗𒁀𒀀 𒇽𒃲𒆷𒋙\n𒀀𒈾 𒇽...,[Šulā son of] Ṣillā of the Ilia family has [vo...,157,148
2,borsippa_P202504,borsippa,P202504,Legal,Ahiya’ūtu Archive,Borsippa,893964,Mix,achaemenid,-547,...,,"Adapted from Caroline Waerzeggers, The Ezida T...",Please cite this page as http://oracc.org/bors...,[x u₄]-mu tar-din-nu {iti}ŠE TA UD [x]-KAM\n[a...,UNK ūmu tardinnu Addaru ultu ūm UNK\nadi ūm UN...,UNK ūmu tardennu Addaru ištu ūmu UNK\nadi ūmu ...,X 𒌓𒈬 𒋻𒁷𒉡 𒌗𒊺 𒋫 𒌓 X𒄰\n𒀀𒁲 𒌓 𒌋𒐋𒄰 𒐈 𒌓𒈬 𒋾𒅆𒆕𒈨𒌍\n𒌗𒁈 𒌓 ...,[x da]ys tardennu (offering) from [day x until...,247,231
3,borsippa_P202510,borsippa,P202510,Legal,Ibnāya Archive,Borsippa,893964,Mix,achaemenid,-547,...,,"Adapted from Caroline Waerzeggers, The Ezida T...",Please cite this page as http://oracc.org/bors...,1/3 GIN₂ KU₃.BABBAR ina BURU₁₄ GIŠ.ŠUB.BA-šu₂\...,UNK šiqil kaspi ina ebūr isqišu\nša šanat UNK ...,UNK šiqlu kaspu ina ebūru isqu\nša šattu UNK C...,𒑚 𒂆 𒆬𒌓 𒀸 𒂙 𒄑𒊒𒁀𒋙\n𒃻 𒈬 𒁹𒄰 𒁹𒅗𒄠𒁍𒍣𒅀\n𒈗 𒁷𒌁𒆠 𒌉𒋙 𒃻 𒁹𒆪𒊏...,20 š silver of the yield of his prebend of the...,96,96
4,borsippa_P202948,borsippa,P202948,Legal,Ilia Archive,Borsippa,893964,Mix,Neo-Babylonian,-626,...,,"Adapted from Caroline Waerzeggers, The Ezida T...",Please cite this page as http://oracc.org/bors...,3 (PI) 4 BAN₂ ŠE.BAR ŠAM₂ KAŠ.U₂.<SA> SIG₅\nna...,UNK pān UNK sūt uṭṭati šīm billati damqi\nnapt...,UNK pānu UNK sūtu uṭṭatu šīmu billatu damqu\nn...,𒐈 𒉿 𒐉 𒑏 𒊺𒁇 𒉚 𒁉𒌑𒊓 𒅆𒂟\n𒀮𒋫𒉡 𒃻 𒀭𒉺 𒃻 𒁹𒀭𒀝𒁾𒆰\n𒀀𒋙 𒃻 𒁹𒀭...,"0;3.4 barley, the price of prime quality billa...",66,66


## 4. Translations are independent of parsing configuration

The `RunConfig` options that control how broken or damaged signs and words are handled — `max_break_fraction`, `drop_missing`, and `drop_damaged` — operate on the **cuneiform data only**. They affect the transliteration, normalization, lemmatization, and Unicode cuneiform outputs. `mask_pos` as well does not affect the translations, as they are not aligned at the word level and the translations are not POS-tagged.

**Practical consequence:** if you change the default settings of `max_break_fraction`, `drop_missing`, `drop_damaged`, or `mask_pos` — cuneiform text and translation are no longer aligned.

In [6]:
# Demonstrate: strict break filtering vs. full translation
config_strict = RunConfig(fetch_translations=True, max_break_fraction=0.0, limit=5)
records_strict = parse_project(PROJECT, config=config_strict)

# Pick a tablet that has both a translation and some damage
for r_default, r_strict in zip(records, records_strict):
    trans_default = r_default.content.transliterated_str_representation
    trans_strict  = r_strict.content.transliterated_str_representation
    if not r_default.content.english_translation:
        continue
    if trans_default and trans_strict and trans_default.text != trans_strict.text:
        print(f"=== {r_default.metadata.id_text} ===")
        print()
        print("TRANSLITERATION (default config — all words kept):")
        print(trans_default.text[:400])
        print()
        print("TRANSLITERATION (max_break_fraction=0.0 — damaged words replaced with X):")
        print(trans_strict.text[:400])
        print()
        print("ENGLISH TRANSLATION (same in both configs):")
        print(r_default.content.english_translation)
        print()
        assert r_default.content.english_translation == r_strict.content.english_translation
        print("✓ Translation is identical regardless of max_break_fraction")
        break

17:58:49 | INFO    | oracc_parser | Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/borsippa
INFO:oracc_parser:Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/borsippa
Processing borsippa: 100%|██████████| 5/5 [00:00<00:00,  5.93it/s]
17:58:50 | INFO    | oracc_parser | Processed 5 tablets for borsippa from word CSVs
INFO:oracc_parser:Processed 5 tablets for borsippa from word CSVs


=== P202111 ===

TRANSLITERATION (default config — all words kept):
2 1/2 u₄-mu maš-ti-ti ša₂ {iti}GUD 2 1/2 u₄-mu
[maš]-ti-ti {iti}KIN 2 1/2 u₄-mu kap-ri ša₂ {iti}GAN
1 u₄]-mu maš-ti-ti ša₂ {iti}ZIZ₂ 2 u₄-mu maš-ti-ti ša₂ {iti}ŠE
[PAB] 10 1/2 u₄-mu kap-ri u₃ maš-ti-ti
[i]-⸢na⸣ GIŠ.ŠUB.BA ša₂ {m}{d}EN-SUM{na} DUMU-šu₂ ša₂ {m}{d}AG-EDURU-MU DUMU {m}DINGIR-ia₂
[a-di] ṭup-pi-šu₂ a-na e-piš-ša₂-nu-tu i-na pa-ni
{[m}{d]}⸢AMAR⸣.UTU-MU-ib-ni DUMU-šu₂ ša₂ {m}šu-la-a DUMU 

TRANSLITERATION (max_break_fraction=0.0 — damaged words replaced with X):
2 1/2 u₄-mu maš-ti-ti ša₂ {iti}GUD 2 1/2 u₄-mu
X {iti}KIN 2 1/2 u₄-mu kap-ri ša₂ {iti}GAN
X X maš-ti-ti ša₂ {iti}ZIZ₂ 2 u₄-mu maš-ti-ti ša₂ {iti}ŠE
X 10 1/2 u₄-mu kap-ri u₃ maš-ti-ti
X GIŠ.ŠUB.BA ša₂ {m}{d}EN-SUM{na} DUMU-šu₂ ša₂ {m}{d}AG-EDURU-MU DUMU {m}DINGIR-ia₂
X ṭup-pi-šu₂ a-na e-piš-ša₂-nu-tu i-na pa-ni
X DUMU-šu₂ ša₂ {m}šu-la-a DUMU {m}DINGIR-ia₂
X X X {m}{d}AMAR.UTU-MU-ib-ni TA e-su-u₂
X 

ENGLISH TRANSLATION (same in both configs):
2½ days ma

## What's next?

- **Quickstart:** See `01_quickstart.ipynb` to download data, parse a project from CSVs, and export results.
- **Configure parsing:** See `03_configure_and_export.ipynb` for masking PNs, dropping broken signs, and changing output formats.
- **Understand the process:** See `04_oracc_json_processing.ipynb` for how the package processes raw ORACC JSON files and how to add new projects.